# warble → half-arc → stack — talk figure

All examples of the chain **warble → half-arc → stack** from the complex-call training set, on a **shared time axis**. Each call is marked by a solid colored bar above the spectrogram (no translucent overlay).

In [ ]:
# === Setup: load complex-call annotations and extract warble -> half-arc -> stack chains ===
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Folder of <name>.wav + <name>_annotations.csv pairs ---
DATA_DIR = Path(
    "/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/"
    "Pre_processing/das/models/TRAINING_data_complex/latest"
)

REPO_ROOT = Path("/mnt/home/gginosar/repos/gerbil_vocalization_analysis")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vocalization_analysis.spectrogram_viz import read_audio
from vocalization_analysis.spectrogram_viz import plot_spectrogram, COLOR_MAP

# Colors per label (extend the pipeline map with complex-set-only labels).
COLORS = {
    **COLOR_MAP,
    "warble":   "tab:purple",
    "half-arc": "tab:olive",
    "stack":    "tab:blue",
}

MIN_FREQ = 1_000
MAX_FREQ = 60_000
SEQ_THRESHOLD_S = 0.035   # calls within this gap belong to the same chain
TARGET = ["warble", "half-arc", "stack"]


def _load_one(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path).dropna(subset=["start_seconds"]).reset_index(drop=True)
    return df.rename(columns={
        "name": "label", "start_seconds": "onset_s", "stop_seconds": "offset_s",
    })

# Long-form annotation table with file provenance.
_rows = []
for wav in sorted(DATA_DIR.glob("*.wav")):
    csv = wav.with_name(wav.stem + "_annotations.csv")
    if not csv.exists():
        continue
    df = _load_one(csv)
    df["file_stem"] = wav.stem
    df["wav_path"] = str(wav)
    _rows.append(df)
all_ann = pd.concat(_rows, ignore_index=True)

# Build chains (consecutive calls within SEQ_THRESHOLD_S), keeping timing + provenance,
# then keep only chains whose label sequence is exactly warble -> half-arc -> stack.
chains: list[pd.DataFrame] = []
for _stem, df in all_ann.groupby("file_stem", sort=False):
    df = df.sort_values("onset_s").reset_index(drop=True)
    if len(df) == 0:
        continue
    start = 0
    for i in range(1, len(df) + 1):
        if i == len(df):
            split = True
        else:
            gap = float(df.iloc[i]["onset_s"]) - float(df.iloc[i - 1]["offset_s"])
            split = gap > SEQ_THRESHOLD_S
        if split:
            chains.append(df.iloc[start:i].reset_index(drop=True))
            start = i

target_chains = [c for c in chains if list(c["label"]) == TARGET]
print(f"{len(all_ann)} annotations / {all_ann['file_stem'].nunique()} files")
print(f"{len(chains)} chains total, {len(target_chains)} matching {' -> '.join(TARGET)}")


In [ ]:
# === Plot all warble -> half-arc -> stack examples on a shared time axis ===
# - Same xlim across every panel; time is relative to the first call (warble),
#   so x = 0 marks the warble onset.
# - No translucent overlay on the spectrogram; instead a solid colored bar
#   sits just above each call, colored by label.

PAD_S = 0.04             # trailing context (s) after the chain
N_COLS = 4               # panels per row
INCHES_PER_SECOND = 18.0
PANEL_H = 2.0
BAR_LW = 7               # thickness of the label bars
SHOW_STEM = True         # tiny file-stem caption per panel (traceability)

REPO_VIZ_DIR = REPO_ROOT / "data" / "complex_calls_viz" / DATA_DIR.name
REPO_VIZ_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR = DATA_DIR / f"viz_{DATA_DIR.name}"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Common window measured from the warble onset (x = 0): widest chain span +
# trailing padding, shared by every panel so calls are directly comparable.
spans = [float(c["offset_s"].iloc[-1] - c["onset_s"].iloc[0]) for c in target_chains]
WINDOW_S = max(spans) + PAD_S
HEADROOM_KHZ = 0.16 * (MAX_FREQ - MIN_FREQ) / 1000.0   # blank band above spectrogram for bars
Y_BAR = MAX_FREQ / 1000.0 + 0.45 * HEADROOM_KHZ

audio_cache: dict[str, tuple[int, np.ndarray]] = {}
def get_audio(path: str):
    if path not in audio_cache:
        audio_cache[path] = read_audio(Path(path))
    return audio_cache[path]

n = len(target_chains)
n_rows = int(np.ceil(n / N_COLS))
panel_w = INCHES_PER_SECOND * WINDOW_S
fig, axes = plt.subplots(
    n_rows, N_COLS, figsize=(panel_w * N_COLS, PANEL_H * n_rows), squeeze=False
)

for idx, chain in enumerate(target_chains):
    ax = axes[idx // N_COLS, idx % N_COLS]
    fs, x = get_audio(chain["wav_path"].iloc[0])
    dur_s = x.shape[0] / fs

    on0 = float(chain["onset_s"].iloc[0])          # warble onset -> x = 0
    t0 = on0
    t1 = min(dur_s, on0 + WINDOW_S)
    a, b = int(t0 * fs), int(t1 * fs)

    # Spectrogram, drawn in time relative to the warble onset (t = 0).
    mesh = plot_spectrogram(
        ax, x[a:b], fs, min_freq=MIN_FREQ, max_freq=MAX_FREQ, t_start=t0 - on0,
    )
    mesh.set_rasterized(True)

    # Solid label bar above each call.
    for _, row in chain.iterrows():
        ax.plot(
            [float(row["onset_s"]) - on0, float(row["offset_s"]) - on0],
            [Y_BAR, Y_BAR],
            color=COLORS.get(row["label"], "tab:gray"),
            linewidth=BAR_LW, solid_capstyle="butt", clip_on=False,
        )

    ax.set_xlim(0.0, WINDOW_S)
    ax.set_ylim(MIN_FREQ / 1000.0, MAX_FREQ / 1000.0 + HEADROOM_KHZ)
    ax.set_xlabel("Time from warble onset (s)" if idx // N_COLS == n_rows - 1 else "")
    ax.set_ylabel("Frequency (kHz)" if idx % N_COLS == 0 else "")
    if idx % N_COLS != 0:
        ax.tick_params(labelleft=False)
    if idx // N_COLS != n_rows - 1:
        ax.tick_params(labelbottom=False)
    ax.tick_params(labelsize=7)
    if SHOW_STEM:
        ax.set_title(chain["file_stem"].iloc[0][-26:], fontsize=6, loc="left")

# Hide unused axes.
for j in range(n, n_rows * N_COLS):
    axes[j // N_COLS, j % N_COLS].axis("off")

# One shared legend for the three call types.
handles = [
    plt.Line2D([0], [0], color=COLORS[l], lw=BAR_LW, solid_capstyle="butt", label=l)
    for l in TARGET
]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 1.02), fontsize=11)
fig.suptitle(f"warble → half-arc → stack  ({n} examples)", y=1.06, fontsize=13)
plt.tight_layout()

stem = "warble_halfarc_stack_examples"
for d in (VIZ_DIR, REPO_VIZ_DIR):
    for ext in ("png", "pdf"):
        fig.savefig(d / f"{stem}.{ext}", dpi=150, bbox_inches="tight")
print(f"Saved {stem}.{{png,pdf}} -> {VIZ_DIR} and {REPO_VIZ_DIR}")
plt.show()
